## Ripple data analysis

Contains Ripple data collected from Gian over two ss.


In [1]:
# Loading/Import related packages
import sys
import os

#Usual suspects
import pandas as pd
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt

#Extras for plotting
from matplotlib.patches import Patch

#Extra for typing
from collections import defaultdict

# Needed to point the importer to the src folder
sys.path.insert(0,r"C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\Ripple")


#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS

In [2]:
REPO_DIR  = os.path.abspath(os.getcwd())
INPUT_DIR = os.path.join(REPO_DIR, 'data')
OUTPUT_DIR = os.path.join(REPO_DIR, 'results')

combined_data_1_path = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524.pkl')
s1_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524_divided')
s2_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst2_20260826_135844_divided')

result_pandas = pd.read_pickle(combined_data_1_path)
print(result_pandas.keys())
print(result_pandas['trial_metadata']['items'])
# print(result_pandas.items())



dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])
[{'index': 0, 'model': 'fist.glb', 'animation': 0, 'repetitions': 8}, {'index': 1, 'model': 'fasttripodpinch.glb', 'animation': 0, 'repetitions': 8}, {'index': 2, 'model': 'fastfingerext.glb', 'animation': 0, 'repetitions': 8}]


#### Task Parameters

In [98]:
# This task used 3 tasks, each with 8 repitions
n_reps = 8
n_tasks = 3
movement_names = ['fist', 'fast_tripod', 'fast_finger_ext']

samp_freq = 2000
n_channels = 32

In [ ]:
import glob 

def get_task_data(s_dir, n_reps, movement_names, file_type, sampling_freq, win=None):
    """
    Load EMG data from multiple pickle files, 
    then combine repitions per movement type, with the option of specifying a window within each repition to keep.

    Parameters:
    - s_dir: str, directory where the pickle files are located.
    - n_reps: int, number of repetitions per movement.
    - movement_names: list of str, names of movements sorted in the order of collection in task.
    - file_type: str, the type of files to load (e.g., 'move' for movement files, or 'rest' for rest files).
    - sampling_freq: float, sampling frequency of data collection in Hz.
    - win: float, optional, window size in seconds to extract from the middle of the movement
      duration. If None, the entire rep duration is used.

    Returns:
    - iso_dict: dict containing each movement as keys, containing:
        - emg_reps: list of dataframes, each containing the EMG data for a single repetition
        - combined_emg_reps: ndarray, concatenated (wide) EMG data for all repetitions
        - durations: list of floats, duration of each repetition
    """
    task_files = sorted(glob.glob(os.path.join(s_dir, f'*{file_type}*')))

    half_win = win / 2 if win is not None else None
    iso_dict = {}

    for movement_num, movement_type in enumerate(movement_names):
        start = movement_num * n_reps
        end = start + n_reps
        movement_files = task_files[start:end]

        movement_data = [pd.read_pickle(file_name) for file_name in movement_files]
        durations = [rep_data['duration_s'] for rep_data in movement_data]

        rep_emg = []
        for rep_data, dur in zip(movement_data, durations):
            rep_df = pd.DataFrame(rep_data['data'])

            if win is None:
                rep_data = rep_df  # no windowing, take entire rep duration
            else:
                iso_start = int(((dur / 2) - half_win) * sampling_freq)
                iso_end = int(((dur / 2) + half_win) * sampling_freq)
                rep_data = rep_df.iloc[:, iso_start:iso_end]

            rep_emg.append(rep_data)

        iso_dict[movement_type] = {
            'emg_reps': rep_emg,
            'combined_emg_reps': pd.concat(rep_emg, ignore_index=True, axis=1).to_numpy(),
            'durations': durations
        }

    return iso_dict

#### Signal

In [143]:
s1_full = get_task_data(s1_folder, 8, movement_names, 'move', samp_freq, win = None)
s1_iso4 = get_task_data(s1_folder, 8, movement_names, 'move', samp_freq, win = 4)

for key in s1_full.keys():
    s1_fist, s1_ft, s1_ffe = s1_full[key]['combined_emg_reps'], s1_full[key]['combined_emg_reps'], s1_full[key]['combined_emg_reps']

for key in s1_iso4.keys():
    s1_fist_iso4, s1_ft_iso4, s1_ffe_iso4 = s1_iso4[key]['combined_emg_reps'], s1_iso4[key]['combined_emg_reps'], s1_iso4[key]['combined_emg_reps']

#### Noise

In [142]:
s1_rest = get_task_data(s1_folder, 8, ['rest'], 'rest', samp_freq, win = None) # pass the movement_names as only 'rest' to get all rest data
s1_rest_emg = s1_rest['rest']['combined_emg_reps']


### Preprocessing


#### Remove Spikes

In [135]:
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


def remove_spikes_dict(data_dict, threshold_std=3):
    """
    Remove large spikes from EMG data in dictionary. 
    Can remove spikes from each repetition separately or from the combined EMG repetitions.

    Paramters:
    - data_dict: dict, containing each movement as keys, and 'emg_reps', 'combined_emg_reps', 'durations' as values
    - threshold_std: float, number of standard deviations for spike thresholding as required by remove_spikes function

    Returns:
    - clean_dict: dict, containing each movement as keys, and clean emg data and spike masks as values.
    """

    clean_dict = {}

    for movement, data in data_dict.items():

        # include if you want to clean each repitition separately:
        # emg_reps_clean = []
        # spike_mask_reps = []

       
        # for rep_df in data['emg_reps']:
        #     rep_clean, rep_mask = remove_spikes(rep_df.to_numpy(), threshold_std)
        #     emg_reps_clean.append(pd.DataFrame(rep_clean, index=rep_df.index, columns=rep_df.columns))
        #     spike_mask_reps.append(rep_mask)
        
        combined_clean, combined_mask = remove_spikes(data['combined_emg_reps'], threshold_std)


        clean_dict[movement] = {
            **data,  # keep original emg_reps, combined_emg_reps, durations
            # 'emg_reps_clean': emg_reps_clean, # uncomment for cleaning each repitition separately
            # 'spike_mask_reps': spike_mask_reps,
            'combined_emg_reps_clean': combined_clean,
            'spike_mask_combined': combined_mask,
        }

    return clean_dict

#### Check Signal Quality and Noise Levels

In [147]:
from scipy.signal import welch

def calc_PSD(sig, fsamp=2048, nperseg=2048, noverlap=2048/2):
    '''
    Compute the power spectral density for each channel using Welch's method.

    Args:
        sig (ndarray): Multi-channel signal (Channels x Samples)
        fsamp (float): Sampling rate in Hz
        nperseg (int): Number of data points per segment
        overlap (int): Number of overlapping samples
    '''

    f, _ = welch(sig[0,:], fs=fsamp, nperseg=nperseg, noverlap=noverlap)

    P = np.zeros((sig.shape[0], f.shape[0]))

    for i in range(sig.shape[0]):
        _ , P[i,:] = welch(sig[i,:], fs=fsamp, nperseg=nperseg, noverlap=noverlap)

    return P, f

def emg_inspect(move_dict, noise_dict, samp_freq, emg_name_key='combined_emg_reps'):
    """
    Collect variables to inspect EMG data quality, including power spectral density (PSD), power, and signal-to-noise ratio (SNR).

    Parameters:
    - move_dict: dict, containing each movement as keys, with combined EMG repetitions as values
    - noise_dict: dict, containing noise data as keys, with combined EMG repetitions as values
    - samp_freq: int, sampling frequency
    - emg_name_key: str, key to access the combined EMG repetitions in the dictionaries (
        default is 'combined_emg_reps', can be changed to 'combined_emg_reps_clean' if using cleaned data

    Returns:
    - inspect_dict: dict, containing each movement and noise as keys, with PSD, power, and SNR values for inspection
    """
    inspect_dict = {}

    for noise_key in noise_dict.keys():
        noise = noise_dict[noise_key][emg_name_key]
        P_n, f_n = calc_PSD(noise, fsamp=samp_freq, nperseg=samp_freq, noverlap=samp_freq/2)
        p_noise = np.mean(noise**2)
        rms_noise = p_noise**0.5
        P_mean = np.mean(P_n, axis=0)
        P_tot = np.sum(P_mean)
        
        inspect_dict[noise_key] = {
            'psd': (P_n, f_n),
            'power': p_noise,
            'P_tot': P_tot,
            'rms_noise': rms_noise
        }

    for sig_key in move_dict.keys():
        sig = move_dict[sig_key][emg_name_key]
        P_s, f_s = calc_PSD(sig, fsamp=samp_freq, nperseg=samp_freq, noverlap=samp_freq/2)
        p_sig = np.mean(sig**2)
        

        inspect_dict[sig_key] = {
            'psd': (P_s, f_s),
            'power': p_sig,
            'snr': {}
        }

        for noise_key in noise_dict.keys():
            p_noise = inspect_dict[noise_key]['power']
            snr = 10 * np.log10(p_sig / p_noise)
            inspect_dict[sig_key]['snr'][noise_key] = snr

    return inspect_dict



In [149]:
s1_clean = remove_spikes_dict(s1_full, threshold_std=3)
s1_iso4_clean = remove_spikes_dict(s1_iso4, threshold_std=3)
rest_clean = remove_spikes_dict(s1_rest, threshold_std=3)

In [150]:
s1_inspect = emg_inspect(s1_full, s1_rest, samp_freq)
s1_iso4_inspect = emg_inspect(s1_iso4, s1_rest, samp_freq)
s1_clean_inspect = emg_inspect(s1_clean, rest_clean, samp_freq, emg_name_key='combined_emg_reps_clean')
s1_iso4_clean_inspect = emg_inspect(s1_iso4_clean, rest_clean, samp_freq, emg_name_key='combined_emg_reps_clean')


In [ ]:
# Full Clean has best SNR, followed by Iso4 Clean, then Full, then Iso4
print(s1_clean_inspect['fist']['snr'])
print(s1_iso4_clean_inspect['fist']['snr'])
print(s1_inspect['fist']['snr'])
print(s1_iso4_inspect['fist']['snr'])


{'rest': 2.031838893890381}
{'rest': 1.0623535513877869}
{'rest': 0.7192607969045639}
{'rest': 1.887509971857071}


In [157]:
s1_fist_clean = s1_iso4_clean['fist']['combined_emg_reps_clean']
fn = s1_clean_inspect['rest']['psd'][1]
fs = s1_clean_inspect['fist']['psd'][1]
Pn = s1_clean_inspect['rest']['psd'][0]
Ps = s1_clean_inspect['fist']['psd'][0]
# Plot a few channels from the recording together with the power spectrum
channels = np.arange(13,25)
t = np.linspace(0, (s1_fist_clean.shape[1]-1)/samp_freq, s1_fist_clean.shape[1]) # data.shape[1] = n_samples

fig, ax = plt.subplots(1,3, figsize=(18,6))

# Full signal
for i, ch_idx in enumerate(channels):
    trace = s1_fist_clean[ch_idx, :]
    ax[0].plot(t,trace + 1 * i, lw=0.5)
# ax[0].plot(t,requested_path/3, lw=2, color="gray")
# ax[0].plot(t,performed_path/3, lw=1, color="black")    
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("Amplitude + offset (mV)")
ax[0].set_ylim(-100, len(channels) + 100)
ax[0].set_title("Raw EMG signals")

for i, ch_idx in enumerate(channels):
    trace = s1_fist_clean[ch_idx, :]
    ax[1].plot(t,trace + 1 * i, lw=0.5)  
ax[1].set_xlabel("Time (s)")
ax[1].set_ylabel("Amplitude + offset (mV)")
ax[1].set_ylim(-100, len(channels) + 100)
ax[1].set_xlim(24.9,25.1)
ax[1].set_xticks([24.9, 25, 25.1])
ax[1].set_title("Raw EMG signals (zoom)")
# Plot the PSD
ax[2].semilogy(fn, np.ones(len(fn)) * 5.21e-8, color = "darkgreen", linestyle="dashed", lw=1)
ax[2].semilogy(fn, np.ones(len(fn)) * 20.83e-8, color = "green", linestyle="dashed", lw=1)
ax[2].semilogy(fn, np.ones(len(fn)) * 46.87e-8, color = "orange", linestyle="dashed", lw=1)
ax[2].semilogy(fn, np.ones(len(fn)) * 83.33e-8, color = "red", linestyle="dashed", lw=1)
ax[2].semilogy(fn,Pn.T, lw=0.1, color=[0.7, 0.7, 1])
ax[2].semilogy(fs,Ps.T, lw=0.1, color=[1, 0.7, 0.7])
ax[2].semilogy(fn,np.percentile(Pn, 50, axis=0), lw=1, color="blue", label="noise")
ax[2].semilogy(fs,np.percentile(Ps, 50, axis=0), lw=1, color="red", label="signal")
ax[2].set_xlabel("Frequency (Hz)")
ax[2].set_ylabel("PSD (mV$^2$/Hz)")
#x[2].set_ylim(1e-9, 1e-2)
ax[2].legend()
ax[2].set_title("PSD of the raw data")
plt.show()
plt.savefig("ripple_analysis_0909.png", dpi=300, bbox_inches='tight')

In [101]:
#Important: if you are in a jupyter notebook,
#this line allows the plots to be displayed in a separate window instead of inline
#which is needed for the mask and channel reviews gui
%matplotlib qt

from src.utils.select_windows import select_windows
from src.utils.review_channels import review_channels

good_mask_path = "demo_good_mask.npy"
mask_path = "demo_mask.npy"
exclude_path = "demo_exclude.npy"

In [108]:
s1_inspect = emg_inspect(s1_fist_clean, s1_rest_clean, samp_freq)


AttributeError: 'numpy.ndarray' object has no attribute 'keys'

In [107]:
s1_fist_clean, spikes = remove_spikes(s1_fist, threshold_std=3)
s1_fist_iso4_clean, spikes = remove_spikes(s1_fist_iso4, threshold_std=3)
s1_rest_clean, spikes = remove_spikes(s1_rest_emg, threshold_std=3)



In [16]:
review_channels(s1_fist_clean, good_mask_path, fs=samp_freq, label="demo")


seed: auto RMS-outlier mask (32/32 good, bad [])
seed: auto RMS-outlier mask (32/32 good, bad [])


In [21]:
review_channels(s1_fist_iso4_clean, good_mask_path, fs=samp_freq, label="demo")

seed: auto RMS-outlier mask (32/32 good, bad [])


In [14]:
# Custom modules
from src.muniverse.algorithms.decomposition import decompose_cbss

# Load the baseline configuration
with open(os.path.join(REPO_DIR, 'src', 'configs', 'cbss.json')) as f:
    cbss_config = json.load(f)['Config']

# The JSON stores disabled options as the string "None", turn them back into real None
cbss_config = {k: (None if v == "None" else v) for k, v in cbss_config.items()}

# # Fewer iterations than the config asks for, again just to keep the notebook quick
# cbss_config['ica_n_iter'] = 30

# no MUs originally found, lower exp:
cbss_config['opt_function_exp'] = 2

print(json.dumps(cbss_config, indent=2))

{
  "start_time": 0,
  "end_time": -1,
  "sampling_frequency": 2048,
  "bandpass": null,
  "bandpass_order": 2,
  "notch_frequency": 50,
  "notch_n_harmonics": 3,
  "notch_order": 2,
  "notch_width": 1,
  "ext_fact": 16,
  "whitening_method": "ZCA",
  "whitening_reg": "auto",
  "ica_n_iter": 100,
  "opt_initalization": "random",
  "opt_function_exp": 2,
  "opt_max_iter": 100,
  "opt_tol": 0.0001,
  "source_deflation": "gram-schmidt",
  "peel_off": true,
  "fr_peeloff": true,
  "cluster_method": "kmeans",
  "random_seed": 1909,
  "refinement_loop": true,
  "sil_th": 0.85,
  "cov_th": 0.35,
  "verbose_mode": true
}


In [22]:

#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS
# Everything from step 1 to step 8, in one call
decomposer = CBSS(**cbss_config)

sources_cbss, spikes_cbss, sil_cbss, filters_cbss, Z_cbss, centroids_cbss = decomposer.decompose(
    s1_fist_iso4_clean, fsamp=samp_freq
    
)

print(f"\nCBSS found {sources_cbss.shape[0]} motor units, ")

[INFO] Extending signals by factor 16...

STARTING CBSS DECOMPOSITION
Max iterations: 100, Silhouette threshold: 0.850, CoV threshold: 0.350

Refinement loop for source 0 (initial Sil: 0.799, CoV: 1.109, Spikes: 14)
FR peel-off enabled
Refinement loop for source 1 (initial Sil: 0.783, CoV: 1.773, Spikes: 26)
FR peel-off enabled
Refinement loop for source 2 (initial Sil: 0.900, CoV: 1.228, Spikes: 19)
FR peel-off enabled
Refinement loop for source 3 (initial Sil: 0.670, CoV: 1.462, Spikes: 26)
FR peel-off enabled
Refinement loop for source 4 (initial Sil: 0.780, CoV: 1.978, Spikes: 32)
FR peel-off enabled
Refinement loop for source 5 (initial Sil: 0.842, CoV: 1.245, Spikes: 26)
FR peel-off enabled
Refinement loop for source 6 (initial Sil: 0.838, CoV: 1.387, Spikes: 20)
FR peel-off enabled
Refinement loop for source 7 (initial Sil: 0.928, CoV: 2.507, Spikes: 185)
FR peel-off enabled
FR peel-off enabled
Refinement loop for source 9 (initial Sil: 0.946, CoV: 2.539, Spikes: 179)
FR peel-of